# Demo: enhancing VoiceBank-DEMAND utterances

This notebook loads a trained checkpoint, enhances a few test utterances, plays them and shows
spectrograms + per-utterance metrics. Run from the repository root after `pip install -e .`.

In [ ]:
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Audio, display
from se.inference import load_model, enhance
from se.data import load_wav
from se.metrics import all_metrics

CKPT = Path("runs/a00_baseline/checkpoints/best.pt")     # any trained run
DATA = Path("data/voicebank_demand_16k")
device = "cuda" if torch.cuda.is_available() else "cpu"
model, stft, cfg = load_model(CKPT, device)
sr = cfg.data.sample_rate

In [ ]:
names = ["p232_001", "p232_005", "p257_010"]   # pick any test files
results = {}
for n in names:
    clean = load_wav(DATA / "clean_testset_wav" / f"{n}.wav", sr)
    noisy = load_wav(DATA / "noisy_testset_wav" / f"{n}.wav", sr)
    est = enhance(model, stft, torch.from_numpy(noisy), sr).numpy()
    results[n] = (clean, noisy, est)
    m_noisy, m_est = all_metrics(clean, noisy, sr), all_metrics(clean, est, sr)
    print(f"{n}: PESQ {m_noisy['pesq']:.2f} -> {m_est['pesq']:.2f} | "
          f"STOI {m_noisy['stoi']:.3f} -> {m_est['stoi']:.3f} | SI-SDR {m_noisy['si_sdr']:.1f} -> {m_est['si_sdr']:.1f} dB")

In [ ]:
n = names[0]
clean, noisy, est = results[n]
for title, x in [("noisy", noisy), ("enhanced", est), ("clean", clean)]:
    print(title); display(Audio(x, rate=sr))
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
for ax, (title, x) in zip(axes, [("noisy", noisy), ("enhanced", est), ("clean", clean)]):
    ax.specgram(x + 1e-9, NFFT=512, Fs=sr, noverlap=384, cmap="magma", vmin=-120); ax.set_title(title)
plt.tight_layout()

## Comparing ablation runs on one utterance
Point `RUNS` at several trained runs to hear what each component contributes
(e.g. `a05_mag_only` keeps the noisy phase and usually sounds rougher).

In [ ]:
RUNS = ["a00_baseline", "a01_no_metric_disc", "a05_mag_only", "p02_pcs"]
clean, noisy, _ = results[names[0]]
for r in RUNS:
    ck = Path("runs") / r / "checkpoints" / "best.pt"
    if not ck.exists():
        print("missing", ck); continue
    m, s, _ = load_model(ck, device)
    y = enhance(m, s, torch.from_numpy(noisy), sr).numpy()
    print(r, {k: round(v, 3) for k, v in all_metrics(clean, y, sr).items() if k in ("pesq", "stoi", "si_sdr")})
    display(Audio(y, rate=sr))